# EfficientNetB3 — Semi-supervised training trên Colab

Notebook này chạy **độc lập trong `ai/semi_supervised`**, không import hay gọi pipeline trong `ai/grading`.

- Model khởi tạo từ `best_EfficientNetB3_rgb_crop_v1.keras` trên Google Drive.
- Dữ liệu có nhãn luôn được tải từ Kaggle dataset `sehastrajits/fundus-aptosddridirdeyepacsmessidor`.
- Dữ liệu không nhãn luôn được tải từ Kaggle dataset `griffchristenson/unlabeled-retinal-image-dataset`.
- Teacher tạo pseudo-label; student fine-tune trên ảnh thật có nhãn và ảnh pseudo-label.
- Artifact tốt nhất được lưu tại `/content/drive/MyDrive/best_semi_EfficientNetB3.keras`.

Trước khi chạy, chọn **Runtime > Change runtime type > T4 GPU** và thêm Colab Secret `KAGGLE_API_TOKEN`.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('Hãy chọn Runtime > Change runtime type > T4 GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
print('TensorFlow:', tf.__version__)
print('GPU:', gpus)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Clone source và cài dependencies

Notebook chỉ cài dependency cần cho pipeline Keras semi-supervised. TensorFlow đã có sẵn trong Colab.


In [ ]:
import os, subprocess, sys
from pathlib import Path

GITHUB_USERNAME = 'Bang334'
GITHUB_REPO = 'dr-diagnostic-system'
GITHUB_BRANCH = 'feat/merged-dataset-training'
REPO_DIR = Path('/content') / GITHUB_REPO
REPO_URL = f'https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GITHUB_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pandas>=2.0', 'Pillow>=10.0', 'ipywidgets>=8.1', 'kaggle>=2.2.2'
], check=True)
print(f'Đã sẵn sàng tại {REPO_DIR}')


## Checkpoint và hai Kaggle dataset cố định

Notebook tự dò checkpoint khởi tạo `best_EfficientNetB3_rgb_crop_v1.keras` trên Google Drive và luôn tải đúng hai dataset từ Kaggle:

- Có nhãn: `sehastrajits/fundus-aptosddridirdeyepacsmessidor`.
- Không nhãn: `griffchristenson/unlabeled-retinal-image-dataset`.

Dataset có nhãn phải chứa `train/0`, ..., `train/4` (hoặc `training/0`, ..., `training/4`). `val` và `test` không được đưa vào training. Artifact tốt nhất được ghi tại `MyDrive/best_semi_EfficientNetB3.keras`.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

DRIVE_ROOT = Path('/content/drive/MyDrive')
ARTIFACT_PATH = DRIVE_ROOT / 'best_semi_EfficientNetB3.keras'
CHECKPOINT_NAME = 'best_EfficientNetB3_rgb_crop_v1.keras'
LABELED_KAGGLE_DATASET = 'sehastrajits/fundus-aptosddridirdeyepacsmessidor'
UNLABELED_KAGGLE_DATASET = 'griffchristenson/unlabeled-retinal-image-dataset'

checkpoint_candidates = []
for current_root, _, files in os.walk(DRIVE_ROOT):
    if CHECKPOINT_NAME in files:
        checkpoint_candidates.append(str(Path(current_root) / CHECKPOINT_NAME))
DEFAULT_CHECKPOINT_PATH = str(DRIVE_ROOT / CHECKPOINT_NAME)
checkpoint_candidates = list(dict.fromkeys(sorted(checkpoint_candidates) + [DEFAULT_CHECKPOINT_PATH]))

checkpoint_widget = widgets.Combobox(
    options=checkpoint_candidates,
    value=checkpoint_candidates[0],
    description='Checkpoint:',
    ensure_option=False,
    layout=widgets.Layout(width='95%'),
)
display(checkpoint_widget)
print('Labeled Kaggle  :', LABELED_KAGGLE_DATASET)
print('Unlabeled Kaggle:', UNLABELED_KAGGLE_DATASET)
print('Artifact        :', ARTIFACT_PATH)


## Tải hai dataset từ Kaggle và tạo manifest ảnh có nhãn

Cell này bắt buộc tải hai Kaggle dataset cố định ở trên. Dữ liệu được cache trong `/content/selected_data`; manifest chỉ quét split train và không phụ thuộc `ai/grading`.


In [ ]:
import getpass, shutil, zipfile
import pandas as pd
from google.colab import userdata

IMAGE_EXTENSIONS = {'.bmp', '.jpeg', '.jpg', '.png', '.tif', '.tiff', '.webp'}

def get_kaggle_token():
    try:
        token = userdata.get('KAGGLE_API_TOKEN')
    except Exception:
        token = getpass.getpass('D?n Kaggle API token: ').strip()
    if not token:
        raise RuntimeError('Ch?a cung c?p KAGGLE_API_TOKEN')
    os.environ['KAGGLE_API_TOKEN'] = token

def download_kaggle_source(dataset_ref, extract_name):
    target = Path('/content/selected_data') / extract_name
    marker = target / '.kaggle_source'
    if marker.is_file() and marker.read_text(encoding='utf-8').strip() == dataset_ref:
        print(f'D?ng l?i dataset Kaggle ?? t?i: {target}')
        return target.resolve()
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True)
    get_kaggle_token()
    kaggle_cli = [sys.executable, '-m', 'kaggle']
    subprocess.run(
        kaggle_cli + ['datasets', 'files', '-d', dataset_ref, '--page-size', '20'],
        check=True, capture_output=True, text=True,
    )
    print(f'?ang t?i Kaggle dataset {dataset_ref}...')
    subprocess.run(kaggle_cli + ['datasets', 'download', '-d', dataset_ref, '-p', str(target)], check=True)
    archives = sorted(target.glob('*.zip'))
    if not archives:
        raise FileNotFoundError(f'Kaggle kh?ng t?o ZIP trong {target}')
    for archive_path in archives:
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(target)
        archive_path.unlink()
    marker.write_text(dataset_ref, encoding='utf-8')
    return target.resolve()

def find_labeled_train_dir(dataset_root):
    dataset_root = Path(dataset_root).resolve()
    candidates = [dataset_root]
    candidates.extend(
        path for path in dataset_root.rglob('*')
        if path.is_dir() and path.name.lower() in {'train', 'training'}
    )
    for candidate in candidates:
        class_dirs = {
            path.name: path for path in candidate.iterdir()
            if path.is_dir() and path.name in {'0', '1', '2', '3', '4'}
        }
        if set(class_dirs) == {'0', '1', '2', '3', '4'}:
            return candidate, class_dirs
    raise ValueError(f'Kh?ng t?m th?y train/0..4 ho?c training/0..4 b?n d??i {dataset_root}')

def create_labeled_manifest(dataset_root, csv_path):
    train_dir, class_dirs = find_labeled_train_dir(dataset_root)
    records = []
    for label in range(5):
        for image_path in sorted(class_dirs[str(label)].rglob('*')):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append({'image_path': str(image_path.resolve()), 'diagnosis': label})
    frame = pd.DataFrame(records)
    if frame.empty:
        raise ValueError(f'Kh?ng t?m th?y ?nh c? nh?n trong {train_dir}')
    missing = sorted(set(range(5)) - set(frame['diagnosis'].unique()))
    if missing:
        raise ValueError(f'Train thi?u ?nh cho c?c l?p: {missing}')
    frame.to_csv(csv_path, index=False)
    return train_dir, frame

CHECKPOINT_PATH = Path(checkpoint_widget.value).expanduser().resolve()
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'Checkpoint kh?ng t?n t?i: {CHECKPOINT_PATH}')

LABELED_ROOT = download_kaggle_source(LABELED_KAGGLE_DATASET, 'labeled_dataset')
UNLABELED_DIR = download_kaggle_source(UNLABELED_KAGGLE_DATASET, 'unlabeled_dataset')

OUTPUT_DIR = ARTIFACT_PATH.parent
LABELED_CSV = OUTPUT_DIR / 'labeled_train.csv'
TRAIN_DIR, labeled_df = create_labeled_manifest(LABELED_ROOT, LABELED_CSV)
unlabeled_count = sum(
    1 for path in UNLABELED_DIR.rglob('*')
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)
if unlabeled_count == 0:
    raise ValueError(f'Kh?ng t?m th?y ?nh ch?a nh?n trong {UNLABELED_DIR}')

SEMI_CONFIG = {
    'threshold': 0.95,
    'pseudo_weight': 0.25,
    'epochs': 10,
    'batch_size': 16,
    'lr': 1e-4,
}
print(f'Checkpoint     : {CHECKPOINT_PATH}')
print(f'Labeled train : {TRAIN_DIR} ({len(labeled_df):,} ?nh)')
print(f'Unlabeled     : {UNLABELED_DIR} ({unlabeled_count:,} ?nh)')
print(f'Output model  : {ARTIFACT_PATH}')
print('C?u h?nh      :', SEMI_CONFIG)
display(labeled_df.groupby('diagnosis').size().rename('images').to_frame())


## Train semi-supervised độc lập

Artifact tốt nhất được lưu trực tiếp tại `/content/drive/MyDrive/best_semi_EfficientNetB3.keras`. Nếu file đã tồn tại, hãy đổi tên hoặc xóa file cũ trước khi train một run mới.


In [ ]:
from ai.semi_supervised.keras_semi_supervised import train_keras_semi_supervised

if ARTIFACT_PATH.exists():
    raise RuntimeError(f'Artifact đã tồn tại: {ARTIFACT_PATH}. Hãy đổi tên hoặc xóa file cũ để train mới.')

best_model_path = train_keras_semi_supervised(
    model_path=str(CHECKPOINT_PATH),
    labeled_csv=str(LABELED_CSV),
    unlabeled_dir=str(UNLABELED_DIR),
    output_dir=str(OUTPUT_DIR),
    threshold=SEMI_CONFIG['threshold'],
    pseudo_weight=SEMI_CONFIG['pseudo_weight'],
    epochs=SEMI_CONFIG['epochs'],
    batch_size=SEMI_CONFIG['batch_size'],
    lr=SEMI_CONFIG['lr'],
    input_size=(300, 300),
)
if Path(best_model_path).resolve() != ARTIFACT_PATH.resolve():
    raise RuntimeError(f'Artifact được lưu sai đường dẫn: {best_model_path}')
print('Checkpoint semi tốt nhất:', best_model_path)


## Kiểm tra artifact sau khi train

`pseudo_labels.csv` dùng để audit ảnh được teacher chấp nhận; `history.json` lưu loss/accuracy theo epoch.


In [ ]:
import json

pseudo_path = OUTPUT_DIR / 'pseudo_labels.csv'
history_path = OUTPUT_DIR / 'history.json'
for artifact in (pseudo_path, history_path, ARTIFACT_PATH):
    if not artifact.exists():
        raise FileNotFoundError(f'Thiếu artifact: {artifact}')

pseudo_df = pd.read_csv(pseudo_path)
history = json.loads(history_path.read_text(encoding='utf-8'))
print(f'Pseudo-label được giữ: {len(pseudo_df):,}')
if not pseudo_df.empty:
    display(pseudo_df.groupby('pseudo_label').agg(images=('image_path', 'size'), mean_confidence=('confidence', 'mean')))
    display(pseudo_df.head(20))
print(json.dumps(history, indent=2, ensure_ascii=False))
